# Extract keypoints

Indexes `datasets/`, runs jazz's palm detector and landmark net over every image,
and writes `keypoints.npz` for `experiment_mlp.ipynb`.

`palm_rois.npz` and `landmarks.npz` are resumable caches of the two slow passes,
keyed on model, device and the exact file list.

In [11]:
import re
import string
from pathlib import Path

DATASETS = Path("datasets")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LETTERS = set(string.ascii_uppercase)
CLASSES = sorted(LETTERS)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
Z_CLIPS = set(range(81, 147)) | set(range(177, 258)) | set(range(380, 397))
_F4_KEEP = re.compile(r"^\d+\.jpe?g$|_test\.jpe?g$", re.I)
_F10_LABEL = re.compile(r"^([A-Za-z])\d+_")


def images_in(folder):
    """Image files directly inside `folder`, in stable order.

    Skips AppleDouble sidecars (`._name.jpg`), which source 5 ships thousands of
    under __MACOSX and which are 4 KB metadata blobs, not decodable images.
    """
    return sorted(p for p in folder.iterdir()
                  if p.suffix.lower() in IMAGE_EXTS and not p.name.startswith("._"))


def letter_folders(root, keep=LETTERS):
    """(path, LABEL) for every <LETTER> folder at any depth under `root`.

    Depth-agnostic, so train/test/valid wrappers are folded into one set.
    """
    for d in sorted(p for p in root.rglob("*") if p.is_dir() and p.name.upper() in keep):
        if "__MACOSX" in d.parts:
            continue
        for img in images_in(d):
            yield img, d.name.upper()


def source_3():
    """Class folders are named 0-27; 0-24 are A-Y and the rest are not letters."""
    for i, letter in enumerate(string.ascii_uppercase[:25]):
        for img in images_in(DATASETS / "3/data" / str(i)):
            yield img, letter


def source_4():
    """Only `<number>.jpg` and `<something>_test.jpg`; the rest are augmentations."""
    for img, label in letter_folders(DATASETS / "4/ASL_Alphabet_Dataset/asl_alphabet_train"):
        if _F4_KEEP.search(img.name):
            yield img, label


def source_5():
    yield from letter_folders(DATASETS / "5")


def source_6():
    yield from letter_folders(DATASETS / "6")


def source_8():
    yield from letter_folders(DATASETS / "8")


def source_9():
    yield from letter_folders(DATASETS / "9/SignAlphaSet/SignAlphaSet")


def source_10():
    """Roboflow export: flat folders with no class dirs, but the letter is the
    filename prefix (U7_jpg.rf.<hash>.jpg -> U), which matches the one-hot
    _classes.csv on all 1728 rows, so the csv is redundant."""
    for d in sorted(p for p in (DATASETS / "10").iterdir() if p.is_dir()):
        for img in images_in(d):
            m = _F10_LABEL.match(img.name)
            if m:
                yield img, m.group(1).upper()


def source_11():
    yield from letter_folders(DATASETS / "11", LETTERS - {"Z"})


def source_12():
    yield from letter_folders(DATASETS / "12/extracted/root/Root/Type_01_(Raw_Gesture)",
                              LETTERS - {"J"})


def source_video_frames():
    """Frames extracted from video sources 15 (J, Z) and 16 (all letters)."""
    for img, label in letter_folders(DATASETS / "video_frames"):
        src, rest = img.name.split("_", 1)
        if src == "16":
            yield img, label
        elif src == "15":
            letter, clip, _ = rest.split("_", 2)
            if letter == "J" or int(clip) in Z_CLIPS:
                yield img, label


SOURCES = {
    "3": source_3, "4": source_4, "5": source_5, "6": source_6, "8": source_8,
    "9": source_9, "10": source_10, "11": source_11, "12": source_12,
    "video_frames": source_video_frames,
}

all_samples = []                                # (path, LABEL, source)
for name, load in SOURCES.items():
    # Sources 6 and 8 each ship a byte-identical copy of themselves in a subfolder
    # (6/asl_dataset/asl_dataset, 8/asl-alphabet-test), so drop repeated filenames.
    seen = set()
    for path, label in load():
        key = (label, path.name)
        if label in LETTERS and key not in seen:
            seen.add(key)
            all_samples.append((path, label, name))

print(f"{len(all_samples):,} images indexed from {len(SOURCES)} sources")

KeyboardInterrupt: 

In [ ]:
# --- Stage 1 of jazz's hand pipeline: palm detection -> rotated hand ROI --------
# The 192 detector locates the palm; its box plus two of its keypoints define a
# square ROI, rotated hand-up, exactly as jazz's GetHandRoiFromPalm builds it.
# Weights are jazz's 192 detector (tflite -> onnx -> torch), batched on GPU.
# Stage 2 (landmarks) is separate; these crops are its input format.
import math
from pathlib import Path

import cv2
import numpy as np
import torch

IMG_SIZE = 224
HAND_MODELS = Path("_hand_models")
PALM_MODEL = "palm_detection_192_full.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DET_SIZE = 192
PALM_PAD = 0.3          # shrink inside the detector canvas; 0.0/0.15/0.3 measured
#                         77% / 83% / 86% detection, flat past 0.3
ROI_SCALE = 3.0                 # jazz HandDetectionConfig.roiSizeMultipleOfPalm
ROI_OFFSET = 0.33               # jazz HandDetectionConfig.roiOffsetFromPalm
UPRIGHT = (0.0, -1.0)           # jazz HandPoseDetectionEngine.uprightVec
BRANCHES = (1152, 864)          # anchors per branch; batched output is regrouped on these

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| opencv", cv2.__version__)


def ssd_anchors(input_size=DET_SIZE, strides=(8, 16, 16, 16), offset=0.5):
    """The 2016 SsdAnchorsCalculator centres the palm boxes are regressed against."""
    anchors, n, layer = [], len(strides), 0
    while layer < n:
        last, per_cell = layer, 0
        while last < n and strides[last] == strides[layer]:
            per_cell += 2                  # one aspect ratio + one interpolated scale
            last += 1
        fm = math.ceil(input_size / strides[layer])
        for y in range(fm):
            for x in range(fm):
                anchors += [((x + offset) / fm, (y + offset) / fm)] * per_cell
        layer = last
    return np.array(anchors, np.float32)


ANCHORS = ssd_anchors()
_DETECTOR = None


def palm_detector():
    """The TorchScript detector, loaded once onto DEVICE and reused."""
    global _DETECTOR
    if _DETECTOR is None:
        _DETECTOR = torch.jit.load(str(HAND_MODELS / PALM_MODEL)).to(DEVICE).eval()
    return _DETECTOR


def letterbox(bgr, size, pad=0.0):
    """Aspect-preserving fit into a size x size canvas, shrunk by `pad`.

    The anchors only cover boxes up to ~75% of the frame, so a hand that fills a
    tight crop stays invisible to the detector until it is padded down.
    """
    h, w = bgr.shape[:2]
    s = size / (1.0 + pad) / max(h, w)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    canvas = np.zeros((size, size, 3), np.uint8)
    ox, oy = (size - nw) // 2, (size - nh) // 2
    canvas[oy:oy + nh, ox:ox + nw] = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_AREA)
    return canvas, s, ox, oy


def palm_input(bgr):
    """The 192 tensor for one image, plus what is needed to map results back to it."""
    canvas, scale, ox, oy = letterbox(bgr, DET_SIZE, PALM_PAD)
    # Resize/warp/crop are per-channel, so channel order only matters here, at the
    # model boundary. Converting the 192 canvas is cheaper than the full image.
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0, scale, ox, oy


def regroup(flat, n):
    """(1, n*2016, C) branch-major -> (n, 2016, C) back on the host."""
    parts, off = [], 0
    for size in BRANCHES:
        parts.append(flat[0, off:off + n * size].reshape(n, size, -1))
        off += n * size
    return torch.cat(parts, 1).cpu().numpy()


def palm_batch(bgrs):
    """Stage 1 over a list of images -> (N, 5) of score, cx, cy, side, degrees.

    One forward pass for the whole list, which is the point of being on the GPU.
    """
    prepared = [palm_input(b) for b in bgrs]
    x = torch.from_numpy(np.stack([p[0] for p in prepared])).to(DEVICE)
    with torch.no_grad():
        outs = palm_detector()(x)
    # Picked by trailing dim rather than position: 18 box floats and one logit each.
    boxes = regroup(next(t for t in outs if t.shape[-1] == 18), len(bgrs))
    logits = regroup(next(t for t in outs if t.shape[-1] == 1), len(bgrs))[..., 0]
    return np.array([decode_palm(b, lg, *p[1:])
                     for b, lg, p in zip(boxes, logits, prepared)], np.float32)


def decode_palm(boxes, logits, scale, ox, oy):
    """Winning anchor -> (score, cx, cy, side, degrees) in original-image pixels."""
    best = int(logits.argmax())                       # sigmoid is monotonic
    score = 1.0 / (1.0 + math.exp(-float(np.clip(logits[best], -100, 100))))
    b, (ax, ay) = boxes[best], ANCHORS[best]

    def to_px(u, v):                                  # canvas-normalised -> image px
        return ((u * DET_SIZE - ox) / scale, (v * DET_SIZE - oy) / scale)

    cx, cy = to_px(b[0] / DET_SIZE + ax, b[1] / DET_SIZE + ay)
    box_w = b[2] / scale                              # jazz sizes the ROI off width
    # GetHandRoiFromPalm: orientation is middle-finger MCP minus thumb CMC, turned
    # onto UPRIGHT, then the box centre slides along that vector by 0.33 * width.
    mx, my = to_px(b[8] / DET_SIZE + ax, b[9] / DET_SIZE + ay)     # keypoint 2
    tx, ty = to_px(b[14] / DET_SIZE + ax, b[15] / DET_SIZE + ay)   # keypoint 5
    vx, vy = mx - tx, my - ty
    norm = math.hypot(vx, vy) or 1.0
    hx, hy = vx / norm, vy / norm

    dot = max(-1.0, min(1.0, hx * UPRIGHT[0] + hy * UPRIGHT[1]))
    deg = math.degrees(math.acos(dot)) * (1.0 if UPRIGHT[0] * hy - UPRIGHT[1] * hx > 0 else -1.0)
    cx += hx * ROI_OFFSET * box_w
    cy += hy * ROI_OFFSET * box_w
    return score, cx, cy, box_w * ROI_SCALE, deg


def crop_roi(bgr, cx, cy, side, deg, size=IMG_SIZE):
    """The ROI square as a size x size RGB image: one warp does rotate+scale+shift."""
    m = cv2.getRotationMatrix2D((cx, cy), deg, size / side)
    m[0, 2] += size / 2 - cx
    m[1, 2] += size / 2 - cy
    warped = cv2.warpAffine(bgr, m, (size, size), flags=cv2.INTER_LINEAR)
    return cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)

In [ ]:
# Stage 1 runs over everything ONCE: images with no palm are thrown away, and only the
# survivors get split, so train/val stay balanced over real data.
import hashlib
import random
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

BATCH_SIZE = 64
RESUME_PALM_CACHE = True    # False = ignore palm cache and rescan from scratch
MIN_PALM_SCORE = 0.15           # jazz HandDetectionConfig.palmConfidenceThreshold
ROI_CACHE = Path("palm_rois.npz")
SCAN_BATCH = 256                # images per forward pass
CHECKPOINT = 20_000             # rows between cache writes


def scan_palms(items):
    """(N, 5) array of score, cx, cy, side, degrees -- cached, since it is the slow pass.

    Resumable. The cache records how many rows are finished and is rewritten however
    the loop exits, so interrupting the kernel costs at most the batch in flight;
    rerun the cell and it picks up from there. Unreadable images keep their zero row,
    so a score of 0 drops them below.
    """
    paths = [str(p) for p, _, _ in items]
    # Model, device and the exact file list are all in the key. Any of them changing
    # changes the rows -- cpu and gpu differ by ~0.3% on the boxes -- and silently
    # resuming across that would poison the run.
    key = hashlib.sha256("\n".join([PALM_MODEL, DEVICE, *paths]).encode()).hexdigest()

    out, start = np.zeros((len(paths), 5), np.float32), 0
    if RESUME_PALM_CACHE and ROI_CACHE.exists():
        cached = np.load(ROI_CACHE)
        if str(cached["key"]) == key:
            out, start = cached["rois"], int(cached["done"])
    if start >= len(paths):
        return out

    def save(done):
        np.savez(ROI_CACHE, key=key, rois=out, done=done)

    done = saved = start
    try:
        with tqdm(total=len(paths), initial=start, desc="palm scan", unit="img") as bar:
            for done in range(start, len(paths), SCAN_BATCH):
                chunk = paths[done:done + SCAN_BATCH]
                images = [cv2.imread(p) for p in chunk]
                ok = [i for i, im in enumerate(images) if im is not None]
                if ok:
                    for i, roi in zip(ok, palm_batch([images[i] for i in ok])):
                        out[done + i] = roi
                bar.update(len(chunk))
                if done - saved >= CHECKPOINT:
                    save(done)
                    saved = done
        done = len(paths)
    finally:
        save(done)
    return out


rois = scan_palms(all_samples)
detected = [(p, lab, src, *roi[1:]) for (p, lab, src), roi in zip(all_samples, rois)
            if roi[0] >= MIN_PALM_SCORE]

by_class = defaultdict(list)
for item in detected:
    by_class[item[1]].append(item)

rng = random.Random(0)
train_samples, val_samples = [], []
for label in CLASSES:
    items = by_class.pop(label)
    rng.shuffle(items)
    n_val = int(len(items) * 0.2)
    val_samples += items[:n_val]
    train_samples += items[n_val:]


class SignImages(Dataset):
    """Decodes and crops to the palm ROI found in the scan above."""

    def __init__(self, items, img_size=IMG_SIZE):
        self.items = items
        self.img_size = img_size

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label, _, cx, cy, side, deg = self.items[i]
        bgr = cv2.imread(str(path))
        if bgr is None:
            raise FileNotFoundError(f"unreadable image: {path}")
        rgb = crop_roi(bgr, cx, cy, side, deg, self.img_size)
        return torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0, CLASS_TO_IDX[label]


train_dataset = SignImages(train_samples)
val_dataset = SignImages(val_samples)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        pin_memory=torch.cuda.is_available())

print(f"{len(all_samples)} indexed -> {len(detected)} with a palm "
      f"({len(detected) / len(all_samples):.1%}) -> "
      f"{len(train_samples)} train / {len(val_samples)} val")

In [ ]:
# Eyeball one batch: every crop should be a hand, upright and roughly the same size.
# A batch is already (N, 3, 224, 224), which is the landmark model's input shape, so
# the keypoint pass is one batched invoke per batch whenever it gets added.
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))
print(f"batch {tuple(images.shape)} {images.dtype}  "
      f"range {images.min():.2f}-{images.max():.2f}")

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[lab], fontsize=10)
    ax.axis("off")
plt.tight_layout()

In [ ]:
# --- Stage 2: the crops -> 21 hand keypoints ----------------------------------
# jazz's default landmark net, batched on GPU (native batching, no regroup).
import hashlib
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

RESUME_KP_CACHE = True      # False = ignore landmark cache and re-extract
LM_MODEL = "hand_landmark_224_full.pt"
LM_SIZE = 224                   # net input; crops must already be this size
NUM_KP = 21
WORLD_SCALE = 13.0              # hand-pose baked this into the world head Gemm
KP_CACHE = Path("landmarks.npz")
_KP_FIELDS = ("kp", "world", "presence", "handed")

_LANDMARKER = None


def landmarker():
    """The TorchScript landmark net, loaded once onto DEVICE and reused."""
    global _LANDMARKER
    if _LANDMARKER is None:
        _LANDMARKER = torch.jit.load(str(HAND_MODELS / LM_MODEL)).to(DEVICE).eval()
    return _LANDMARKER


def landmarks_of(images):
    """(N, 3, 224, 224) in [0,1] -> keypoints, world, presence, raw handedness.

    Head order is the MediaPipe one the onnx declares and hand-pose's loader
    confirms: Identity, Identity_1, Identity_2, Identity_3 = landmarks, presence,
    handedness, world. Both coordinate heads come out of that onnx pre-scaled, so
    undo it here; what is left is exactly what jazz reads off the tflite.
    """
    with torch.no_grad():
        out = landmarker()(images.permute(0, 2, 3, 1).to(DEVICE))   # the net wants NHWC
    return (out[0].reshape(-1, NUM_KP, 3).mul(LM_SIZE).cpu().numpy(),
            out[3].reshape(-1, NUM_KP, 3).div(WORLD_SCALE).cpu().numpy(),
            out[1].reshape(-1).cpu().numpy(),
            out[2].reshape(-1).cpu().numpy())


def jazz_handedness(raw):
    """jazz's rule, MediaPipeHandLandmarkDetector.cs: round(1 - raw), Left=0 Right=1.

    Kept as a function over the raw score rather than baked into the arrays, because
    the label is only as trustworthy as the mirroring: jazz reads a selfie camera, and
    a dataset shot un-mirrored flips the answer. The raw score is what gets cached.
    """
    return np.rint(1.0 - raw).astype(np.int8)


def extract_keypoints(dataset):
    """Run stage 2 over a whole dataset, in order, so row i is that dataset's item i."""
    n = len(dataset)
    kp = np.zeros((n, NUM_KP, 3), np.float32)
    world = np.zeros((n, NUM_KP, 3), np.float32)
    presence = np.zeros(n, np.float32)
    handed = np.zeros(n, np.float32)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        pin_memory=torch.cuda.is_available())
    at = 0
    for images, _ in tqdm(loader, desc="landmarks", unit="batch"):
        batch = landmarks_of(images)
        for dst, src in zip((kp, world, presence, handed), batch):
            dst[at:at + len(src)] = src
        at += len(batch[0])
    return kp, world, presence, handed


# Same cache rule as the palm scan: model, device and the exact file list are the key.
_KP_KEY = hashlib.sha256("\n".join(
    [LM_MODEL, DEVICE, *_KP_FIELDS] + [str(i[0]) for i in train_samples + val_samples]
).encode()).hexdigest()

if RESUME_KP_CACHE and KP_CACHE.exists() and str(np.load(KP_CACHE)["key"]) == _KP_KEY:
    _z = np.load(KP_CACHE)
    train_kp, train_world, train_presence, train_handed = (
        _z["train_kp"], _z["train_world"], _z["train_presence"], _z["train_handed"])
    val_kp, val_world, val_presence, val_handed = (
        _z["val_kp"], _z["val_world"], _z["val_presence"], _z["val_handed"])
else:
    train_kp, train_world, train_presence, train_handed = extract_keypoints(train_dataset)
    val_kp, val_world, val_presence, val_handed = extract_keypoints(val_dataset)
    np.savez(KP_CACHE, key=_KP_KEY,
             train_kp=train_kp, train_world=train_world,
             train_presence=train_presence, train_handed=train_handed,
             val_kp=val_kp, val_world=val_world,
             val_presence=val_presence, val_handed=val_handed)

print(f"train {train_kp.shape}  val {val_kp.shape}")
print(f"keypoints x {train_kp[..., 0].min():7.1f} to {train_kp[..., 0].max():7.1f} px "
      f"(crop is {LM_SIZE})")
print(f"world     xyz {train_world.min():+.3f} to {train_world.max():+.3f}")
print(f"presence  {train_presence.mean():.3f} mean, "
      f"{(train_presence < 0.5).sum():,} of {len(train_presence):,} below 0.5")
print(f"handed    {(jazz_handedness(train_handed) == 1).mean():.1%} right by jazz's rule, "
      f"raw {train_handed.min():.2f} to {train_handed.max():.2f}")

In [ ]:
# Eyeball stage 2: the skeleton should sit on the hand, not float beside it.
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

FINGERS = [(0, 1, 2, 3, 4), (0, 5, 6, 7, 8), (0, 9, 10, 11, 12),
           (0, 13, 14, 15, 16), (0, 17, 18, 19, 20)]

images, labels = next(iter(DataLoader(val_dataset, batch_size=6, shuffle=True)))
kp, world, presence, handed = landmarks_of(images)
side = ["L", "R"]

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, img, lab, pts, pres, hand in zip(axes, images, labels, kp, presence,
                                         jazz_handedness(handed)):
    ax.imshow(img.permute(1, 2, 0).numpy())
    for finger in FINGERS:
        ax.plot(pts[list(finger), 0], pts[list(finger), 1], "-", lw=1.2, c="lime")
    ax.scatter(pts[:, 0], pts[:, 1], s=6, c="red", zorder=3)
    ax.set_title(f"{CLASSES[lab]}  {side[hand]}  p={pres:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()

In [ ]:
import numpy as np

np.set_printoptions(suppress=True, precision=4)
train_kp[1006]

In [10]:
# --- Test: a separate capture, left/<LETTER> and right/<LETTER> -------------------
# Every step is the one train and val went through, reused rather than restated: the
# same palm gate, the same presence and handedness gates.
import hashlib
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

TEST_ROOT = DATASETS / "signlanguagetestdata"   # left/<LETTER>/ and right/<LETTER>/
KEYPOINTS = Path("keypoints.npz")               # the handoff experiment_mlp.ipynb reads

test_samples = [(p, lab, hand) for hand in ("left", "right")
                for p, lab in letter_folders(TEST_ROOT / hand)]
_TEST_KEY = hashlib.sha256("\n".join(
    [PALM_MODEL, LM_MODEL, DEVICE] + [str(p) for p, _, _ in test_samples]
).encode()).hexdigest()


def scan_rois(paths):
    """Stage 1 over a path list, no cache -> (N, 5). Unreadable images keep a zero row."""
    out = np.zeros((len(paths), 5), np.float32)
    for at in tqdm(range(0, len(paths), SCAN_BATCH), desc="palm scan", unit="batch"):
        images = [cv2.imread(str(p)) for p in paths[at:at + SCAN_BATCH]]
        ok = [i for i, im in enumerate(images) if im is not None]
        if ok:
            for i, roi in zip(ok, palm_batch([images[i] for i in ok])):
                out[at + i] = roi
    return out


def degrees_of(items):
    """The angle crop_roi rotated away, one per row. It is the last field of each item."""
    return np.array([item[-1] for item in items], np.float32)


_cached = np.load(KEYPOINTS) if KEYPOINTS.exists() else None
if _cached is not None and str(_cached["test_key"]) == _TEST_KEY and "test_deg" in _cached:
    test_kp, test_world, test_presence, test_handed = (
        _cached["test_kp"], _cached["test_world"],
        _cached["test_presence"], _cached["test_handed"])
    test_labels, test_left = _cached["test_labels"], _cached["test_left"]
    test_deg = _cached["test_deg"]
    test_detected = int(_cached["test_detected"])
else:
    rois = scan_rois([p for p, _, _ in test_samples])
    items = [(p, lab, hand, *roi[1:]) for (p, lab, hand), roi in zip(test_samples, rois)
             if roi[0] >= MIN_PALM_SCORE]       # jazz's palmConfidenceThreshold
    test_kp, test_world, test_presence, test_handed = extract_keypoints(SignImages(items))
    test_labels = np.array([CLASS_TO_IDX[lab] for _, lab, *_ in items], np.int64)
    test_left = np.array([hand == "left" for _, _, hand, *_ in items])
    test_deg = degrees_of(items)
    test_detected = len(items)

np.savez(KEYPOINTS, test_key=_TEST_KEY, classes=np.array(CLASSES),
         train_kp=train_kp, train_world=train_world, train_presence=train_presence,
         train_handed=train_handed, train_deg=degrees_of(train_samples),
         train_labels=np.array([CLASS_TO_IDX[lab] for _, lab, *_ in train_samples], np.int64),
         val_kp=val_kp, val_world=val_world, val_presence=val_presence,
         val_handed=val_handed, val_deg=degrees_of(val_samples),
         val_labels=np.array([CLASS_TO_IDX[lab] for _, lab, *_ in val_samples], np.int64),
         test_kp=test_kp, test_world=test_world, test_presence=test_presence,
         test_handed=test_handed, test_deg=test_deg,
         test_labels=test_labels, test_left=test_left,
         test_labelled=len(test_samples), test_detected=test_detected)

print(f"{len(test_samples):,} labelled -> {test_detected:,} past the palm gate")
print(f"test {test_kp.shape}   left {test_left.mean():.1%} by folder")
print(f"deg   {test_deg.min():.0f} to {test_deg.max():.0f} degrees, "
      f"{(np.abs(test_deg) > 30).mean():.1%} of test turned more than 30")
print(f"saved -> {KEYPOINTS}  "
      f"(train {len(train_kp):,} / val {len(val_kp):,} / test {len(test_kp):,})")

landmarks: 100%|██████████| 102/102 [01:11<00:00,  1.42batch/s]


6,576 labelled -> 6,526 past the palm gate
test (6526, 21, 3)   left 50.5% by folder
deg   -171 to 179 degrees, 50.6% of test turned more than 30
saved -> keypoints.npz  (train 287,851 / val 71,950 / test 6,526)
